# **4.7 Forward Propagation, Backward Propagation, and Computational Graphs**
Automatic computation of gradients (automatic differentiation) has greatly simplified the implementation of deep learning algorithms. Before automatic differentiation, even small tweaks to complex models required manually recalculating complicated derivatives, and academic papers had to dedicate lots of pages to deriving update rules. In this section, we’ll dive into the details of backpropagation through some basic math and computation graphs. First, we’ll focus on a single-hidden-layer multilayer perceptron with weight decay ($L_2$ regularization).

## **4.7.1 Forward Propagation**
Forward propagation (forward propagation or forward pass) refers to calculating and storing the results of each layer in a neural network in order, from the input layer to the output layer.<br><br>
We will study the mechanism of a single hidden layer neural network step by step. For simplicity, let's assume the input sample is $x \in \mathbb{R}^d$, and our hidden layer does not include a bias term. The intermediate variables here are:
##### **(4.7.1)**
$$z=W^{(1)}x$$
Here, $W^{(1)} \in \mathbb{R}^{h×d}$ is the weight parameter of the hidden layer. After passing the intermediate variable $z \in \mathbb{R}^h$ through the activation function $\phi$, we get a hidden activation vector of length h:
##### **(4.7.2)**
$$h=\phi(z)$$
The hidden variable $h$ is also an intermediate variable. Suppose the output layer's parameters only include the weight $W^{(2)} \in \mathbb{R}^{(q×h)}$, we can get the output layer variable, which is a vector of length $q$:
##### **(4.7.3)**
$$o=W^{(2)}h$$
Let $l$ be the loss function and $y$ the sample label; we can calculate the loss term for a single data sample,
##### **(4.7.4)**
$$L=l(o,y)$$
According to the definition of $L_2$ regularization, given a hyperparameter $λ$, the regularization term is
##### **(4.7.5)**
$$s=\frac{\lambda}{2}\left(||W^{(1)}||^{2}_{F}+||W^{(2)}||^{2}_{F}\right)$$
The Frobenius norm of a matrix is the $L_2$ norm applied after the matrix is flattened into a vector. Finally, the regularized loss of the model on a given data sample is:
##### **(4.7.6)**
$$J=L+s$$
In the discussion that follows, we will refer to J as the objective function.

## **4.7.2 Forward Propagation Computation Graph**
Drawing a computation graph helps us visualize the dependencies between operators and variables in a computation. [Figure 4.7.1](#fig471) shows the computation graph corresponding to the simple network described above, where squares represent variables and circles represent operators. The bottom-left corner represents the input, and the top-right corner represents the output. Note that the arrows indicating data flow point primarily to the right and upward.
##### **Fig(4.7.1)**
![Forward Propagation Computation Graph](../../images/picture4_7_1.png)

## **4.7.3 Backpropagation**
Backward propagation (also known as backpropagation) refers to a method for computing the gradients of neural network parameters. In short, this method traverses the network in reverse order—from the output layer to the input layer—based on the chain rule of calculus. The algorithm stores any intermediate variables (partial derivatives) required to compute the gradients of certain parameters. Suppose we have functions $Y = f(X)$ and $Z = g(Y)$, where the inputs and outputs $X$, $Y$, and $Z$ are tensors of arbitrary shape. Using the chain rule, we can compute the derivative of $Z$ with respect to $X$.
##### **(4.7.7)**
$$\frac{\partial Z}{\partial X}=prod\left(\frac{\partial Z}{\partial Y},\frac{\partial Y}{\partial X}\right)$$
Here, we use the `prod` operator to multiply its arguments after performing the necessary operations (such as transposition and swapping input positions). For vectors, this is straightforward—it is simply matrix-vector multiplication. For high-dimensional tensors, we use the appropriate corresponding terms. The `prod` operator encompasses all of these notations.<br><br>
Recall that the parameters of the simple single-hidden-layer network shown in [Figure 4.7.1](#fig471) are $W^{(1)}$ and $W^{(2)}$. The purpose of backpropagation is to compute the gradients $\partial J/ \partial W^{(1)}$ and $\partial J/ \partial W^{(2)}$. To do this, we apply the chain rule to compute the gradients of each intermediate variable and parameter in turn. The order of computation is the reverse of that performed during forward propagation, because we need to start from the results of the computational graph and work backward toward the parameters. The first step is to compute the gradients of the objective function $J = L + s$ with respect to the loss term $L$ and the regularization term $s$.
##### **(4.7.8)**
$$\frac{\partial J}{\partial L}=1 \quad and \quad \frac{\partial J}{\partial s}=1$$
##### **(4.7.9)**
$$\frac{\partial J}{\partial o}= prod\left(\frac{\partial J}{\partial L},\frac{\partial L}{\partial o}\right)=\frac{\partial L}{\partial o} \in \mathbb{R}^{q}$$
Next, we calculate the gradients of the regularization term with respect to the two parameters:
##### **(4.7.10)**
$$\frac{\partial s}{\partial W^{(1)}}=\lambda W^{(1)} \, and \, \frac{\partial s}{\partial W^{(2)}}=\lambda W^{(2)}$$
Now we can compute the gradient $\partial J/ \partial W(2) \in \mathbb{R}^{q×h}$ of the model parameters closest to the output layer. Using the chain rule, we obtain:
##### **(4.7.11)**
$$\frac{\partial J}{\partial W^{(2)}}=prod\left(\frac{\partial J}{\partial o},\frac{\partial o}{\partial W^{(2)}}\right)+prod\left(\frac{\partial J}{\partial s},\frac{\partial s}{\partial W^{(2)}}\right) = \frac{\partial J}{\partial o}h^{T}+\lambda W^{(2)}$$
To obtain the gradient of $W^{(1)}$, we need to continue backpropagating from the output layer to the hidden layer. The gradient $\partial J/ \partial h \in \mathbb{R}^{h}$ with respect to the hidden layer output is given by the following equation:
##### **(4.7.12)**
$$\frac{\partial J}{\partial h}=prod\left(\frac{\partial J}{\partial o},\frac{\partial o}{\partial h}\right)=W^{(2)}\frac{\partial J}{\partial o}$$
Since the activation function $\phi$ is computed element-wise, calculating the gradient $\partial J/ \partial z \in \mathbb{R}^{h}$ of the intermediate variable $z$ requires the use of the element-wise multiplication operator, which we denote by $\odot$:
##### **(4.7.13)**
$$\frac{\partial J}{\partial z}=prod\left(\frac{\partial J}{\partial h},\frac{\partial h}{\partial z}\right)=\frac{\partial J}{\partial h} \odot \phi '(z)$$
Finally, we can obtain the gradient $\partial J/ \partial W^{(1)} \in \mathbb{R}^{h\times d}$ of the model parameters closest to the input layer. By the chain rule, we obtain:
##### **(4.7.14)**
$$\frac{\partial J}{\partial W^{(1)}}=prod\left(\frac{\partial J}{\partial z},\frac{\partial z}{\partial W^{(1)}}\right)+prod\left(\frac{\partial J}{\partial s},\frac{\partial s}{\partial W^{(1)}}\right)=\frac{\partial J}{\partial z}x^{T}+\lambda W^{(1)}$$

## **4.7.4 Training a Neural Network**
When training a neural network, forward propagation and backward propagation are interdependent. For forward propagation, we traverse the computational graph in the direction of the dependencies and compute all variables along the path. These results are then used in backward propagation, where the order of computation is the reverse of that in the computational graph.<br><br>
Taking the simple network described above as an example: On the one hand, during forward propagation, the calculation of the regularization term [(4.7.5)](#475) depends on the current values of the model parameters W(1) and W(2). These are provided by the optimization algorithm based on the backpropagation from the most recent iteration. On the other hand, during backpropagation, the calculation of the gradient of the parameter [(4.7.11)](#4711) depends on the current value of the hidden variable h, which is provided by forward propagation.<br><br>
Therefore, when training a neural network, after initializing the model parameters, we alternate between forward propagation and backward propagation, using the gradients provided by backward propagation to update the model parameters. Note that backpropagation reuses the intermediate values stored during forward propagation to avoid redundant computations. **One consequence of this is that we need to retain these intermediate values until backpropagation is complete. This is also one of the reasons why training requires more memory (GPU memory) than simple prediction. Furthermore, the size of these intermediate values is roughly proportional to the number of network layers and the batch size. Consequently, using larger batch sizes to train deeper networks is more likely to result in out-of-memory errors**.

## **4.7.5 Summary**
* In the computational graph defined by a neural network, the forward pass computes and stores intermediate variables sequentially, proceeding from the input layer to the output layer.
* Backpropagation calculates and stores the gradients of the neural network's intermediate variables and parameters in the reverse order (from the output layer to the input layer).
* When training deep learning models, forward propagation and backward propagation are interdependent.
* Training requires more memory than prediction.